# Tutorial 5: Audio Generation

This tutorial explores two core topics in audio generation: speech generation (Text-to-Speech, TTS) and music generation. In the first part, we introduce several fundamental and practical toolkits for audio analysis and preprocessing. In the second part, we take a deeper dive into modern TTS systems by implementing the key components of a state-of-the-art model, VALL-E. Finally, we study music generation, covering essential data preprocessing techniques such as source separation, as well as methods for controlled music generation.

## Part 1: Audio Analysis Toolkits
In this part, we will study several key audio preprocessing steps, including data preparation, spectrogram representations, and their visualization.

### **1.1 Data Preprocessing**
In this section, we begin by learning [librosa](https://librosa.org/doc/latest/index.html), a widely used audio processing toolkit with convenient and user-friendly Python APIs.

In [ ]:
!pip install "librosa~=0.10.1" "matplotlib~=3.7.1" pysptk

In [ ]:
import librosa
import librosa.display
import librosa.effects
import librosa.util

import numpy as np
import matplotlib.pyplot as plt

# used to play audio files
import IPython.display as ipd
from base64 import b64decode

We will take our first glance at a handful of raw audio clips from the [HarperValleyBank](https://arxiv.org/abs/2010.13929) dataset. Working with audio files in this way is similar to what you might experience when working with data exported from call center recordings or similar telephone/app-based human-human interactions.

To start, ensure you can execute the commands below to locate and load the audio files.

In [ ]:
# download dataset from public GDrive
!wget -O hvb_sampledata.zip https://drive.usercontent.google.com/download?id=1tFaTu7LWuCxn3DkAyudGGyv6eilVO5pC

In [ ]:
!ls
!unzip -o hvb_sampledata.zip

In [ ]:
ls sample

In the sample folder, we've put a few example conversations between an agent (bank employee) and a caller (bank customer). Each conversation is split into two `.wav` files by speaker. For example, `sample/agent/0002f70f7386445b.wav` and `sample/caller/0002f70f7386445b.wav` belong to the same conversation. The first file contains everything the agent says while the second contains everything the caller says. This speaker-separated format is common in speech corpora.

In [ ]:
ls sample/agent


In [ ]:
ls sample/caller

**Playing audio files.**

We can listen to the audio files to familiarize ourselves with the data. The long pauses you hear between utterances is because each audio file is from only one side of the conversation.

In [ ]:
# agent side of the conversation
ipd.Audio('./sample/agent/0002f70f7386445b.wav')

In [ ]:
# caller side of the conversation
ipd.Audio('./sample/caller/0002f70f7386445b.wav')

**Loading audio into an array.**

When working with spoken language, we often want to visualize and process the raw audio before applying machine learning or signal processing approaches. To do this, we typically work with an audio file as an array or tensor. Our audio files are a single channel (mono, not stereo), or we sometimes have a two-party conversation encoded as stereo with one speaker in the left/right channel for separation.

The `librosa` package contains a suite of utilities to open and process waveforms. The function `librosa.load` loads an audio file as a floating point time series. It will return a numpy array containing samples from the audio clip and the sample rate (`sr`) listed in the audio file header. The sample rate is an integeter for how many samples (array indices) correspond to 1 second of realtime audio. The higher the sample rate, the higher the "resolution" of an audio clip.

In [ ]:
# sr represents a "sample rate"
wav_agent, sr_agent = librosa.load('./sample/agent/0002f70f7386445b.wav')
wav_caller, sr_caller = librosa.load('./sample/caller/0002f70f7386445b.wav')

In [ ]:
wav_agent.shape, sr_agent

In [ ]:
wav_caller.shape, sr_caller

**Merge waveforms.**

To familiarize yourself with waveforms and librosa, the code below demonstrates how to merge the agent and caller waveforms into a single audio signal. We assume that the audio files to be combined always share the same sampling rate (`sr_caller == sr_agent`). If the two waveforms have different lengths, the merged audio should have a length equal to `length = max(length_1, length_2)`, with zero-padding applied to the shorter signal as needed.

In [ ]:
merged_wav = None
sr_merged = sr_agent

# Merge two mono waveforms (mix them) with zero-padding to the same length.
# Assumption: sr_agent == sr_caller (given)

L = max(len(wav_agent), len(wav_caller))
agent_pad = np.zeros(L, dtype=np.float32)
caller_pad = np.zeros(L, dtype=np.float32)

agent_pad[:len(wav_agent)] = wav_agent
caller_pad[:len(wav_caller)] = wav_caller

# Mix (simple sum). Optional: normalize to avoid clipping.
merged_wav = agent_pad + caller_pad
peak = np.max(np.abs(merged_wav)) + 1e-9
if peak > 1.0:
    merged_wav = merged_wav / peak

#############################
ipd.Audio(merged_wav, rate=sr_merged)

**Visualizing audio using time-frequency spectrograms.**

In general, it is hard to directly work with waveforms in machine learning. There are differences in magnitude, variable lengths, and speech-relevant patterns are hard to discern by visualizing a waveform directly. We often rely on signal processing tools to "standardize" waveforms. In speech, it is common to convert raw waveforms to the frequency domain with **spectrograms**. A spectrogram is a time series of short-window fourier transforms, so we can see how the frequencies of speech change over the course of an utterance.

The human auditory system does not perceive all frequencies equally. In very low or high frequencies (in hertz), our ears are less capable at discriminating between different frequencies.

The Mel-scale is a scale of pitches judged by human listeners to be equal in distance one from another. It is roughly linear between 0 and 1000hz and logarithmic above 1000hz, as human ears become less adept at differentiating frequencies. We can think of Mel scale as a 'bin size' of frequencies to match how humans perceive speech. This is helpful when building spoken language systems because it means our feature representations more closely match what a human listener would perceive from the same audio.

We will use this helper function to visualize your merged audio file. Note the keywords used below. First we will set the max frequency considered for our mel scale (`fmax`) to the max possible frequency for this sampling rate (Nyquist Frequency, `sample_rate/2`). The number of bins is effectively the "resolution" in mel scale of the vertical axis.

In [ ]:
# helper function to plot a mel spectrogram
# arguments: (wave array, sampling rate, number of mel bins, max frequency of mel scale)
def plot_melspectrogram(wav, sr, annotations=None, n_mels=256, fmax=4096,
            fig=None, ax=None, show_legend=True):

  if ax == None:
    fig, ax = plt.subplots(1,1,figsize=(20,5))
  M = librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=n_mels, fmax=fmax, n_fft=2048)
  M_db = librosa.power_to_db(M, ref=np.max)
  img = librosa.display.specshow(M_db, y_axis='mel', x_axis='time', ax=ax, fmax=fmax)
  if show_legend:
    ax.set(title='Mel spectrogram display')
    fig.colorbar(img, ax=ax, format="%+2.f dB")

  # iterate over list of text annotations and draw them
  if annotations is not None:
    for x,y,text in annotations:
      ax.annotate(
        text,
        xy=(x,y), xycoords='data',
        xytext=(10, -50), textcoords='offset pixels',
        horizontalalignment='right',
        color='white',
        fontsize=20,
        verticalalignment='bottom',
        arrowprops=dict(arrowstyle= '-|>', color='white', lw=1, ls='-')
        )

Now we can use the helper function to visualize a Mel spectrogram of the combined audio file you created

In [ ]:
plot_melspectrogram(merged_wav, sr_agent, annotations=None, n_mels=256, fmax=sr_agent/2)

###**1.2 Exercise: Silence Removal**
We often analyze utterances from a single speaker at a time, rather than the conversation. We also want to focus on the actual speech audio rather than long silences / pauses between utterances.

Focusing on the **agent**, your task is to remove most of the major silences from the raw waveform file. This will collapse all of the agent's utterances into a single array with just short pauses between utterances. Removing silences in this way helps focus on the speech parts of an audio file. Sometimes this process is called voice activity detection (and is more dificult in scenarios with strong background noise or distortion when speech can be hard to identify from the background).

One way to do this:

Use `librosa.effects.split` to split the agent .wav file by silence.
Clip out the silences and combine the audio back into a single array with silences removed. Ensure you aren't filtering too aggressively and clipping out actual audio.
You are free to try other approaches to silence removal. It should sound like a more or less continuous utterance stream from the speaker, but it need not be perfect. Describe your approach briefly along with the implementation.

In [ ]:
recon_agent = None # TODO: create a NumPy array with silences removed

#############################
#### YOUR CODE GOES HERE ####

# Use librosa to find non-silent intervals, then concatenate them.
# top_db controls what counts as "silence" (higher -> more aggressive removal).
intervals = librosa.effects.split(wav_agent, top_db=30)

# Optional: keep a short pause between utterances (e.g., 50 ms)
pause_len = int(0.05 * sr_agent)
pause = np.zeros(pause_len, dtype=np.float32)

chunks = []
for start, end in intervals:
    chunks.append(wav_agent[start:end])
    chunks.append(pause)

recon_agent = np.concatenate(chunks) if len(chunks) > 0 else wav_agent.copy()



In [ ]:
# listen to your audio file
ipd.Audio(recon_agent, rate=sr_agent)

In [ ]:
# This should now have minimal gaps between utterances
# (leaving some small silences is okay)
plot_melspectrogram(recon_agent, sr_agent)

## Part 2: Speech Generation (Text-to-Speech, TTS)

In this part, we will dive into the architecture of a state-of-the-art LLM-based TTS model, [VALL-E](https://arxiv.org/pdf/2301.02111).

### **2.1 Residual Vector Quantized-Variational AutoEncoder (RVQ-VAE)**

LLMs operate on discrete tokens. Therefore, the first step in applying LLM-style modeling to audio is to **discretize** continuous audio signals into tokens. VALL-E achieves this by using a neural audio codec based on Residual Vector Quantization ([RVQ-VAE](https://arxiv.org/pdf/2107.03312)).

Compared to the standard [VQ-VAE](https://arxiv.org/pdf/1711.00937), RVQ-VAE performs iterative quantization, progressively encoding residual errors across multiple codebooks. This design significantly reduces reconstruction error and improves audio fidelity, making it well suited for high-quality speech generation.

In [ ]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
import torch
torch.rand(2).cuda()


Below is a simple implementation of a Residual Vector Quantizer (RVQ), which consists of multiple codebooks applied sequentially across iterations to progressively quantize the residual error.

In [ ]:
import torch; torch.manual_seed(0)
import torch.nn as nn
import torch.nn.functional as F


class ResidualVectorQuantizer(nn.Module):
    """
    Residual Vector Quantization:
      x0 = x
      for m in [0..M-1]:
         e_m = nearest(codebook_m, x_m)
         x_{m+1} = x_m - e_m
      quantized = sum_m(e_m)

    Inputs/outputs are (B, C, T) where vectors to quantize are along C.
    """
    def __init__(self, dim: int, codebook_size: int = 100, num_quantizers: int = 4, beta: float = 0.25):
        super().__init__()
        self.dim = dim
        self.K = codebook_size
        self.M = num_quantizers
        self.beta = beta

        self.codebooks = nn.ModuleList([nn.Embedding(self.K, self.dim) for _ in range(self.M)])
        for cb in self.codebooks:
            nn.init.uniform_(cb.weight, -1.0 / self.K, 1.0 / self.K)

    @torch.no_grad()
    def _nearest_code(self, x_bt_d: torch.Tensor, cb: nn.Embedding) -> torch.Tensor:
        """
        x_bt_d: (N, D), cb.weight: (K, D)
        returns indices: (N,)
        """
        # Squared Euclidean distance: ||x - e||^2 = ||x||^2 + ||e||^2 - 2 x·e
        x2 = (x_bt_d ** 2).sum(dim=1, keepdim=True)          # (N,1)
        e2 = (cb.weight ** 2).sum(dim=1).unsqueeze(0)        # (1,K)
        xe = x_bt_d @ cb.weight.t()                          # (N,K)
        dist = x2 + e2 - 2 * xe
        return dist.argmin(dim=1)

    def forward(self, x: torch.Tensor):
        """
        x: (B, C, T)
        returns:
          x_q: (B, C, T)
          all_indices: list of (B, T) code indices per quantizer
          vq_loss: scalar (commitment + codebook loss)
        """
        B, C, T = x.shape
        assert C == self.dim, f"Expected C==dim ({self.dim}), got {C}"

        residual = x
        x_q = torch.zeros_like(x)
        all_indices = []
        vq_loss = 0.0

        # reshape to vectors: (B*T, C)
        for m, cb in enumerate(self.codebooks):
            r = residual.permute(0, 2, 1).contiguous().view(B * T, C)  # (N, C)

            with torch.no_grad():
                idx = self._nearest_code(r, cb)  # (N,)
            all_indices.append(idx.view(B, T))

            e = cb(idx).view(B, T, C).permute(0, 2, 1).contiguous()    # (B, C, T)

            # Losses (classic VQ-VAE):
            # - codebook loss: move embeddings toward encoder outputs (stopgrad on encoder)
            # - commitment loss: move encoder outputs toward embeddings (stopgrad on embeddings)
            vq_loss = vq_loss + F.mse_loss(e, residual.detach()) + self.beta * F.mse_loss(residual, e.detach())

            # Straight-through estimator
            e_st = residual + (e - residual).detach()

            x_q = x_q + e_st
            residual = residual - e_st  # residual quantization

        return x_q, all_indices, vq_loss

Below is a complete implementation of Codec, i.e., the audio tokenizer.

In [ ]:
class SimpleCodec(nn.Module):
    """
    A tiny end-to-end neural codec:
      waveform -> encoder -> RVQ -> decoder -> waveform
    """
    def __init__(self, latent_dim=64, codebook_size=100, num_quantizers=4):
        super().__init__()
        # Downsample by 8 total (2*2*2)
        self.enc = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=4, padding=3),  #723
            nn.ReLU(),
            nn.Conv1d(32, 256, kernel_size=7, stride=8, padding=3),  #723
            nn.ReLU(),
            nn.Conv1d(256, latent_dim, kernel_size=7, stride=8, padding=3),  #723
        )
        self.rvq = ResidualVectorQuantizer(dim=latent_dim, codebook_size=codebook_size, num_quantizers=num_quantizers)

        self.dec = nn.Sequential(
            nn.ConvTranspose1d(latent_dim, 256, kernel_size=16, stride=8, padding=7), #823
            nn.ReLU(),
            nn.ConvTranspose1d(256, 32, kernel_size=16, stride=8, padding=7),  #823
            nn.ReLU(),
            nn.ConvTranspose1d(32, 1, kernel_size=8, stride=4, padding=3),  #823
            nn.Tanh(),
        )

    def forward(self, wav):
        z = self.enc(wav)
        z_q, indices, vq_loss = self.rvq(z)
        wav_hat = self.dec(z_q)
        # crop/pad to match length exactly (ConvTranspose can be off by 1-2)
        T = wav.shape[-1]
        wav_hat = wav_hat[..., :T] if wav_hat.shape[-1] >= T else F.pad(wav_hat, (0, T - wav_hat.shape[-1]))
        return wav_hat, vq_loss, indices

    def decode_from_codes(self, codes):  # (B,M,Tc) -> (B,C,Tc)
        B, M, Tc = codes.shape
        assert M == self.rvq.M
        embs = []
        for m, cb in enumerate(self.rvq.codebooks):
            em = cb(codes[:, m, :])  # (B,Tc,C)
            embs.append(em)
        zq = torch.stack(embs, dim=0).sum(dim=0)  # (B,Tc,C)
        zq = zq.permute(0, 2, 1).contiguous()   # (B,C,Tc)
        wav_hat = self.dec(zq)
        return wav_hat


Let's download two audio examples.

In [ ]:
# prepare data
!wget https://huggingface.co/oliveryanzuolu/70113_GenAI_helper_models/resolve/main/Apple.wav
!wget https://huggingface.co/oliveryanzuolu/70113_GenAI_helper_models/resolve/main/Orange.wav
!ls

In [ ]:
import librosa

latent_dim = 512
codebook_size = 100
num_quantizers = 4

lr = 1e-3
num_epochs = 200
intv  = 10

RVQ = SimpleCodec(latent_dim=latent_dim, codebook_size=codebook_size, num_quantizers=num_quantizers).cuda()
opt = torch.optim.Adam(RVQ.parameters(), lr=lr)

wav_apple, sr_apple = librosa.load('./Apple.wav')
wav_orange, sr_orange = librosa.load('./Orange.wav')
print(wav_apple.shape, wav_orange.shape)
wav_apple = torch.from_numpy(wav_apple).unsqueeze(0).unsqueeze(0).float().cuda()
wav_orange = torch.from_numpy(wav_orange).unsqueeze(0).unsqueeze(0).float().cuda()
# Pad short audio sequence
wav = torch.cat([F.pad(wav_apple, (0, wav_orange.shape[-1]-wav_apple.shape[-1])), wav_orange], dim=0)

for i in range(num_epochs):
  wav_hat, vq_loss, _ = RVQ(wav)
  recon_loss = F.mse_loss(wav_hat, wav)
  loss = recon_loss + vq_loss

  opt.zero_grad()
  loss.backward()
  opt.step()

  if (i+1) % intv == 0:
    print(f"Epoch={i} recon_loss={recon_loss.item():.4f}  vq_loss={vq_loss.item():.4f}  total={loss.item():.4f}")

In [ ]:
# Check the reconstruction performance
import IPython.display as ipd

RVQ = RVQ.eval()
with torch.no_grad():
  wav_hat, _, _ = RVQ(wav)
wav_apple_hat = wav_hat[0, 0, :wav_apple.shape[-1]]
wav_orange_hat = wav_hat[1, 0, :wav_orange.shape[-1]]

ipd.Audio(wav_apple.squeeze().detach().cpu().numpy(), rate=sr_apple)

In [ ]:
ipd.Audio(wav_apple_hat.squeeze().detach().cpu().numpy(), rate=sr_apple)

In [ ]:
ipd.Audio(wav_orange.squeeze().detach().cpu().numpy(), rate=sr_orange)

In [ ]:
ipd.Audio(wav_orange_hat.squeeze().detach().cpu().numpy(), rate=sr_orange)

### **2.2 AR and NAR Transformer**
Using RVQ-VAE produces multi-layer discrete tokens. Following VALL-E, we employ an autoregressive (AR) Transformer to predict the first-layer (coarse) tokens, which capture the main structure of the audio, and non-autoregressive (NAR) Transformers to predict the remaining layers in parallel, refining the residual details.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def sinusoidal_positional_encoding(T, D):
    pe = torch.zeros(T, D)
    pos = torch.arange(T).unsqueeze(1)
    div = torch.exp(torch.arange(0, D, 2) * -(math.log(10000.0) / D))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class ARTransformer(nn.Module):
    """
    Autoregressive TransformerEncoder:
        p(c1_t | c1_<t)

    Input:
        c_in: (B, T) teacher-forced (shifted-right) codes
    Output:
        logits: (B, T, V)
    """
    def __init__(self, vocab_size=103, d_model=256, nhead=4, num_layers=4, max_len=25600):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        pe = sinusoidal_positional_encoding(max_len, d_model)
        self.register_buffer('pe', pe)

        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)
        self.bos_id = vocab_size - 3
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self.special_tokens = set([self.bos_id, self.eos_id, self.pad_id])

    def forward(self, c_in, condition):
        B, T = c_in.shape
        T_c = condition.shape[1]
        T += T_c
        tok_emb = torch.cat([condition, self.tok(c_in)], dim=1)
        h = tok_emb + self.pe[None, :T, :].to(tok_emb.device)

        # causal mask enforces AR
        causal = torch.triu(torch.ones(T, T, device=c_in.device), diagonal=1).bool()
        # full attention over conditions
        causal[:T_c] = False
        h = self.tr(h, mask=causal)
        return self.head(h)  # (B,T,V)

    @torch.no_grad()
    def generate(self, T, device, condition, B=1):
        c = torch.full((B, 1), self.bos_id, device=device, dtype=torch.long)
        finished = torch.zeros(B, device=device, dtype=torch.bool)
        for _ in range(T - 1):
            logits = self.forward(c, condition)[:, -1, :]
            nxt = torch.argmax(logits, dim=-1, keepdim=True)
            c = torch.cat([c, nxt], dim=1)
            finished[nxt.squeeze(-1) == self.eos_id] = True
            if torch.all(finished):
              break
        return c  # (B,T)

In [ ]:
class NARTransformer(nn.Module):
    """
    Non-autoregressive TransformerEncoder for ONE residual layer:
        p(cm | c1)

    Conditioning:
        Use coarse codes c1 as the input sequence (embedded), and predict residual codes in parallel.
        (No causal mask)

    Input:
        c1: (B, T)
    Output:
        logits: (B, T, V)
    """
    def __init__(self, vocab_size=103, d_model=256, nhead=4, num_layers=4, num_q_layes=3, max_len=25600):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        pe = sinusoidal_positional_encoding(max_len, d_model)
        self.register_buffer('pe', pe)
        self.layer_emb = nn.Embedding(num_q_layes, d_model)

        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, c1, condition=None, layer_idx=0):
        B, T = c1.shape
        layer_emb = self.layer_emb(torch.full((B, 1), layer_idx, device=c1.device, dtype=torch.long))
        h = torch.cat([condition, layer_emb, self.tok(c1)], dim=1)
        T = h.shape[1]
        h = h + self.pe[None, :T, :].to(h.device)

        h = self.tr(h)  # no mask => parallel
        return self.head(h)  # (B,T,V)

In [ ]:
def compute_losses(ar_model, nar_model, codes, condition):
    """
    codes: (B, M, T)
    nar_models: ModuleList length M-1
    """
    B, M, T = codes.shape
    c1 = codes[:, 0, :]  # (B,T)
    bos_id = ar_model.bos_id
    eos_id = ar_model.eos_id
    pad_id = ar_model.pad_id

    # AR teacher forcing (shift-right)
    bos = torch.full((B, 1), bos_id, device=c1.device, dtype=torch.long)
    eos = torch.full((B, 1), eos_id, device=c1.device, dtype=torch.long)
    c_in = torch.cat([bos, c1], dim=1)
    prefix_len = condition.shape[1]
    logits1 = ar_model(c_in, condition=condition)[:, prefix_len:]
    loss_ar = F.cross_entropy(logits1.reshape(-1, logits1.size(-1)), torch.cat([c1, eos], dim=1).reshape(-1), ignore_index=pad_id)

    # NAR residuals conditioned on coarse
    loss_nar = 0.0
    for m in range(1, M):
        logitsm = nar_model(codes[:, m-1, :], condition=condition, layer_idx=m-1)[:, condition.shape[1]+1:]
        targetm = codes[:, m, :]
        loss_nar = loss_nar + F.cross_entropy(
            logitsm.reshape(-1, logitsm.size(-1)),
            targetm.reshape(-1),
            ignore_index=pad_id
        )

    return loss_ar, loss_nar, loss_ar + loss_nar

In [ ]:
def clean_batch(tokens, bos_id, eos_id, pad_id):
    """
    tokens: LongTensor (B, T)
    returns: list of lists (variable length)
    """
    out = []
    for seq in tokens.tolist():
        seq = seq[1:] if seq and seq[0] == bos_id else seq
        cleaned = []
        for t in seq:
            if t == eos_id:
                break
            if t == pad_id:
                continue
            cleaned.append(t)
        out.append(cleaned)
    return out

def pad_token_lists(token_lists, pad_id=0):
    """
    token_lists: list of lists, e.g. [[1,2,3], [4,5]]
    pad_id: int

    returns:
        LongTensor of shape (B, T_max)
    """
    B = len(token_lists)
    T_max = max(len(seq) for seq in token_lists)

    out = torch.full((B, T_max), pad_id, dtype=torch.long)
    for i, seq in enumerate(token_lists):
        out[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)

    return out

In [ ]:
device = torch.device('cuda')

V = 100
M = 4
B = 2
T = 200
d_model = 256

# pretend these came from RVQ
codes = torch.randint(0, V, (B, M, T), device=device)

# condition
condition = torch.rand((B, 10, d_model)).to(device)

# increase vocab with special tokens
ar = ARTransformer(vocab_size=V+3, d_model=d_model).to(device)
nar = NARTransformer(vocab_size=V+3, d_model=d_model, num_q_layes=M-1).to(device)
bos_id = ar.bos_id
eos_id = ar.eos_id
pad_id = ar.pad_id
special_tokens = ar.special_tokens

loss_ar, loss_nar, total = compute_losses(ar, nar, codes, condition)
total.backward()
print("loss_ar:", float(loss_ar), "loss_nar:", float(loss_nar), "total:", float(total))

# Inference demo: AR -> coarse, then NAR -> residuals
with torch.no_grad():
    condition = torch.rand((1, 10, d_model)).to(device)
    c1_gen = ar.generate(T=T, device=device, B=1, condition=condition)  # (1,T)
    c1_gen = clean_batch(c1_gen, bos_id, eos_id, pad_id)
    c1_gen = pad_token_lists(c1_gen, pad_id).to(device)
    all_codes = [c1_gen]
    for m in range(1, M):
        logits_m = nar(all_codes[-1], condition=condition, layer_idx=m-1)[:, condition.shape[1]+1:]
        all_codes.append(torch.argmax(logits_m, dim=-1))
    all_codes = torch.stack(all_codes, dim=1)  # (1,M,T)
    print("generated codes shape:", all_codes.shape)

### **2.3 Exercise: Build a Toy TTS System**

In this exercise, you will build a toy TTS system, which consists of a CharTokenizer, a training loop for the AR and NAR Transformers, and the inference pipeline for the complete system.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import IPython.display as ipd

# ---- TODO: implement a simple character tokenizer (encode only) ----
class CharTokenizer:
    def __init__(self):
        chars = list("abcdefghijklmnopqrstuvwxyz '.,!?")
        self.pad = 0
        self.bos = 1
        self.eos = 2
        self.stoi = {c: i + 3 for i, c in enumerate(chars)}
        self.vocab_size = 3 + len(chars)

    def encode(self, s: str, max_len: int = 32):
        s = s.lower()
        ids = [self.bos] + [self.stoi.get(c, self.pad) for c in s][:max_len-2] + [self.eos]
        return ids

# ---- Text embedding -> condition tensor ----
class TextConditioner(nn.Module):
    """Turn token ids into a condition tensor (B, L, d_model)."""
    def __init__(self, text_vocab, d_model=256, max_len=256):
        super().__init__()
        self.emb = nn.Embedding(text_vocab, d_model)
        pe = sinusoidal_positional_encoding(max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, text_ids):  # (B, L)
        B, L = text_ids.shape
        h = self.emb(text_ids) + self.pe[None, :L, :].to(text_ids.device)
        return h  # (B, L, d_model)

In [ ]:
# ---- 1) Build a tiny 2-sample "dataset": text -> waveform ----
device = torch.device("cuda")

texts = ["apple", "orange"]
tok = CharTokenizer()
text_ids = pad_token_lists([tok.encode(t) for t in texts], pad_id=tok.pad).to(device)  # (B, L)

# We will reuse the waveforms you loaded earlier (wav_apple, wav_orange) and the padded batch `wav`.
# `wav` was created in Section 2.1 to have shape (2, 1, T_wav).
# If you restarted the runtime, rerun Section 2.1 first.

assert "wav" in globals(), "Please run Section 2.1 first to create `wav` and train `RVQ`."
assert isinstance(RVQ, SimpleCodec), "RVQ should be an instance of SimpleCodec from Section 2.1."

wav = wav.to(device)
B, _, T_wav = wav.shape

# ---- 2) Extract RVQ target codes from waveform ----
RVQ.eval()
with torch.no_grad():
    _, _, codes = RVQ(wav)  # codes: (B, M, Tc)
    codes = torch.stack(codes, dim=1)  # (B, M, Tc)

print("codes shape:", tuple(codes.shape), "(B, M, Tc)")

# ---- 3) Build models: TextConditioner + AR + NAR ----
V = 100                 # codebook size (no special tokens)
d_model = 256
num_q = codes.shape[1]  # M
Tc = codes.shape[2]

# IMPORTANT: ARTransformer expects vocab_size = V + 3 (special tokens are the last 3 ids).
ar = ARTransformer(vocab_size=V + 3, d_model=d_model).to(device)
nar = NARTransformer(vocab_size=V + 3, d_model=d_model, num_q_layes=num_q - 1).to(device)
cond_net = TextConditioner(text_vocab=tok.vocab_size, d_model=d_model).to(device)
ar.train(); nar.train(); cond_net.train()

opt = torch.optim.Adam(list(ar.parameters()) + list(nar.parameters()) + list(cond_net.parameters()), lr=1e-3)

# ---- 4) Training loop (toy) ----
# TODO(Students): try increasing epochs and observe quality.
epochs = 300
print_every = 20

for it in range(1, epochs + 1):
    condition = cond_net(text_ids)  # (B, L, d_model)

    # Pad codes with PAD id? Here both have same Tc, so no need. We keep raw codes.
    # But our Transformer vocabulary includes special tokens in the LAST 3 ids, so codes must be < V.
    # TODO(Students): if you have variable-length Tc, pad with pad_id (ar.pad_id) and set ignore_index properly.

    loss_ar, loss_nar, loss = compute_losses(ar, nar, codes, condition)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if it % print_every == 0:
        print(f"iter={it}  loss_ar={loss_ar.item():.4f}  loss_nar={loss_nar.item():.4f}  total={loss.item():.4f}")

# ---- 5) Inference: text -> codes -> waveform ----
ar.eval(); nar.eval(); cond_net.eval()

max_T = Tc
with torch.no_grad():
    condition = cond_net(text_ids)

    # AR generate coarse codes (length Tc). The model may stop early if it predicts EOS.
    c1_gen = ar.generate(T=max_T, device=device, condition=condition, B=B)  # includes BOS at start
    c1_list = clean_batch(c1_gen, ar.bos_id, ar.eos_id, ar.pad_id)

    # Turn variable-length lists into padded tensor (B, Tc) with PAD
    c1_pad = pad_token_lists(c1_list, pad_id=ar.pad_id).to(device)

    # NAR generate residual layers in parallel
    preds = [c1_pad]
    for layer_idx in range(num_q - 1):
        logits = nar(c1_pad, condition=condition, layer_idx=layer_idx)[:, condition.shape[1] + 1:]
        cm = torch.argmax(logits, dim=-1)
        preds.append(cm)

    pred_codes = torch.stack(preds, dim=1)  # (B, M, Tc)

    # Decode to waveform with the trained codec decoder
    wav_gen = RVQ.decode_from_codes(pred_codes.clamp(0, V-1))



### **2.4 Exercise: TTS API**

Some of you may choose to explore TTS in depth and become experts in this field. Others may simply want to use TTS as a practical tool for their work or for entertainment. For the latter case, it is often unnecessary to build a system from scratch. Below, we demonstrate how to directly call an existing [TTS API](https://github.com/idiap/coqui-ai-TTS) to achieve high-quality speech synthesis with minimal effort.


In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

def generate_mel_spectrogram(audio, sr=16000, n_fft=1024, hop_length=512, n_mels=128,
                             fmin=20, fmax=8000, plot=True, ax=None):
    # calculate mel spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax
    )

    # convert to dB scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    if plot:
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 4))

        img = librosa.display.specshow(
            mel_spec_db,
            x_axis='time',
            y_axis='mel',
            sr=sr,
            fmax=fmax,
            ax=ax
        )
        ax.set_title('Mel Spectrogram')
        plt.colorbar(img, ax=ax, format='%+2.0f dB')

    return mel_spec, mel_spec_db

def visualize_waveform(audio, sr=16000, plot=True, ax=None):
    if plot:
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 3))

        librosa.display.waveshow(audio, sr=sr, ax=ax)
        ax.set_title('Waveform')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Amplitude')

    return audio

In [ ]:
!pip install coqui-tts
!pip install pypinyin


In [ ]:
from TTS.api import TTS
from IPython.display import Audio
import librosa
import torch

In [ ]:
device = torch.device('cuda')
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print(tts.speakers)

In [ ]:
tts.tts_to_file(
  text="Imperial College London",
  speaker="Craig Gutsy",
  language="en",
  file_path="output1.wav"
)
Audio("output1.wav", rate=16000)

In [ ]:
# make sure to run the helper function cell before running this cell
file_path = "output1.wav"
wav, sr = librosa.load(file_path, sr=16000)

fig, axs = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2]})

# generate visualizations
visualize_waveform(wav, sr=sr, ax=axs[0])
generate_mel_spectrogram(wav, sr=sr, ax=axs[1])

plt.tight_layout()
plt.show()

In [ ]:
# Try other languages
tts.tts_to_file(
  text="今天天气怎么样",
  speaker="Craig Gutsy",
  language="zh",
  file_path="output2.wav"
)
Audio("output2.wav", rate=16000)

## Part 3: Music Generation

In this part, we will move further to music generation.

### **3.1 Source Separation**

In music, audio signals are typically composed of multiple sound sources, such as vocals and different instruments. Source separation is therefore a crucial data preprocessing step in many music generation and analysis tasks. In this section, we make use of a deep-learning-based tool, [Demucs](https://github.com/adefossez/demucs), to perform source separation.


In [ ]:
!pip install demucs

In [ ]:
# Import required libraries
import pathlib
import time

import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import librosa
import demucs.separate

# Set up Matplotlib style
plt.rc("axes", linewidth=1.5)
plt.rc("savefig", dpi="150")

In [ ]:
# Separate an audio file into four stems (as MP3 files)
demucs.separate.main(["--mp3", librosa.example("fishin")])

In [ ]:
# Show the audio files
print("Input mixture:")
ipd.display(ipd.Audio(librosa.example("fishin")))

out_dir = pathlib.Path("separated/htdemucs/Karissa_Hobbs_-_Lets_Go_Fishin/")

print("Drums:")
ipd.display(ipd.Audio(out_dir / "drums.mp3"))

print("Bass:")
ipd.display(ipd.Audio(out_dir / "bass.mp3"))

print("Vocals:")
ipd.display(ipd.Audio(out_dir / "vocals.mp3"))

print("Other:")
ipd.display(ipd.Audio(out_dir / "other.mp3"))

In [ ]:
# Show the separation result
fig, ax = plt.subplots(figsize=(9, 10), nrows=5, sharex=True, sharey=True)

out_dir = pathlib.Path("separated/htdemucs/Karissa_Hobbs_-_Lets_Go_Fishin/")

# Load the audio files
y, sr = librosa.load(librosa.example("fishin"), duration=20)
y_drums, sr = librosa.load(out_dir / "drums.mp3", duration=20)
y_bass, sr = librosa.load(out_dir / "bass.mp3", duration=20)
y_vocals, sr = librosa.load(out_dir / "vocals.mp3", duration=20)
y_other, sr = librosa.load(out_dir / "other.mp3", duration=20)

# Compute the power spectrogrms
S = librosa.stft(y)
s_max = np.max(np.abs(S))
S_db = librosa.amplitude_to_db(np.abs(S), ref=s_max)
S_drums_db = librosa.amplitude_to_db(np.abs(librosa.stft(y_drums)), ref=s_max)
S_bass_db = librosa.amplitude_to_db(np.abs(librosa.stft(y_bass)), ref=s_max)
S_vocals_db = librosa.amplitude_to_db(np.abs(librosa.stft(y_vocals)), ref=s_max)
S_other_db = librosa.amplitude_to_db(np.abs(librosa.stft(y_other)), ref=s_max)

# Input mixture
im = librosa.display.specshow(S_db, x_axis="time", y_axis="log", ax=ax[0],
                              vmin=-80, vmax=0)
ax[0].set_title("Input mixture")
ax[0].set_xlabel("")

# Drums
librosa.display.specshow(S_drums_db, x_axis="time", y_axis="log", ax=ax[1],
                         vmin=-80, vmax=0)
ax[1].set_title("Drums")
ax[1].set_xlabel("")

# Bass
librosa.display.specshow(S_bass_db, x_axis="time", y_axis="log", ax=ax[2],
                         vmin=-80, vmax=0)
ax[2].set_title("Bass")
ax[2].set_xlabel("")

# Vocals
librosa.display.specshow(S_vocals_db, x_axis="time", y_axis="log", ax=ax[3],
                         vmin=-80, vmax=0)
ax[3].set_title("Vocals")
ax[3].set_xlabel("")

# Other
librosa.display.specshow(S_other_db, x_axis="time", y_axis="log", ax=ax[4],
                         vmin=-80, vmax=0)
ax[4].set_title("Other")

plt.subplots_adjust(hspace=0.3)
plt.colorbar(im, ax=ax, format="%+2.0f dB")
plt.show()

### **3.2 Text-Music Contrastive Learning**

For controllable music generation, it is essential to incorporate control signals, such as text, into the model in a reasonable manner. In this section, we explore [MuLan](https://arxiv.org/pdf/2208.12415), a [contrastive](https://arxiv.org/pdf/2103.00020) music-text model, in which the text encoder is used to embed textual control signals for music generation.

In [ ]:
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import IPython.display as ipd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3 random music-text pairs (feel free to change the text prompts) ---
pairs = [
    (librosa.example("brahms"),      "classical piano and orchestra"),
    (librosa.example("nutcracker"),  "ballet orchestra, bright and festive"),
    (librosa.example("fishin"),      "guitar and vocals, upbeat folk song"),
]

SR = 16000
DURATION = 6.0   # seconds per clip for the toy CLIP training
N_SAMPLES = int(SR * DURATION)

def load_clip(path, sr=SR, n_samples=N_SAMPLES):
    y, _ = librosa.load(path, sr=sr, duration=DURATION, mono=True)
    if len(y) < n_samples:
        y = np.pad(y, (0, n_samples - len(y)))
    else:
        y = y[:n_samples]
    y = np.clip(y, -1.0, 1.0)
    return y

# Show the 3 audio clips
for p, t in pairs:
    print("Text:", t)
    ipd.display(ipd.Audio(load_clip(p), rate=SR))

# Tokenize text (reuse CharTokenizer from previous sections)
tok = CharTokenizer()
text_tensors = [tok.encode(t, max_len=96) for _, t in pairs]
text_ids = pad_token_lists(text_tensors, pad_id=tok.pad).to(device)

audio_np = np.stack([load_clip(p) for p, _ in pairs], axis=0)  # (B, T)
audio = torch.from_numpy(audio_np).float().unsqueeze(1).to(device)  # (B,1,T)

print("text_ids:", text_ids.shape, "audio:", audio.shape)

In [ ]:
# -----------------------------
# CLIP-like contrastive model
# -----------------------------
class TinyTextEncoder(nn.Module):
    """Char-level text encoder -> single embedding."""
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=3, max_len=128, out_dim=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.proj = nn.Linear(d_model, out_dim)

    def forward(self, ids):  # (B,L)
        B, L = ids.shape
        x = self.emb(ids) + self.pos(torch.arange(L, device=ids.device))[None, :, :]
        # Key padding mask: True means "ignore" (PAD tokens)
        key_padding = (ids == tok.pad)
        h = self.tr(x, src_key_padding_mask=key_padding)  # (B,L,D)
        # Mean pool over non-pad positions
        mask = (~key_padding).float().unsqueeze(-1)       # (B,L,1)
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        return F.normalize(self.proj(pooled), dim=-1)     # (B,out_dim)

class TinyAudioEncoder(nn.Module):
    """Waveform encoder -> single embedding."""
    def __init__(self, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, stride=4, padding=4),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=4, padding=4),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=9, stride=4, padding=4),
            nn.ReLU(),
        )
        self.proj = nn.Linear(128, out_dim)

    def forward(self, wav):  # (B,1,T)
        h = self.net(wav)              # (B,128,T')
        h = h.mean(dim=-1)             # global average pool -> (B,128)
        return F.normalize(self.proj(h), dim=-1)  # (B,out_dim)

class TinyCLIP(nn.Module):
    def __init__(self, vocab_size, emb_dim=256):
        super().__init__()
        self.text_enc = TinyTextEncoder(vocab_size=vocab_size, out_dim=emb_dim)
        self.audio_enc = TinyAudioEncoder(out_dim=emb_dim)
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1/0.07), dtype=torch.float32))

    def forward(self, text_ids, wav):
        t = self.text_enc(text_ids)  # (B,D)
        a = self.audio_enc(wav)      # (B,D)
        scale = self.logit_scale.exp().clamp(1.0, 100.0)
        logits = scale * (t @ a.t()) # (B,B)
        return logits

def clip_loss(logits):
    """Symmetric InfoNCE loss."""
    B = logits.size(0)
    labels = torch.arange(B, device=logits.device)
    loss_t2a = F.cross_entropy(logits, labels)
    loss_a2t = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_t2a + loss_a2t)

clip = TinyCLIP(vocab_size=tok.vocab_size, emb_dim=256).to(device)
opt = torch.optim.AdamW(clip.parameters(), lr=1e-5)

# Train on only 3 pairs (toy demo): it will quickly overfit.
clip.train()
for step in range(600):
    logits = clip(text_ids, audio)
    loss = clip_loss(logits)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (step + 1) % 50 == 0:
        with torch.no_grad():
            acc = (logits.argmax(dim=1) == torch.arange(logits.size(0), device=device)).float().mean().item()
        print(f"step {step+1:03d} | loss={loss.item():.4f} | train-acc={acc:.2f}")


In [ ]:
# After training, we will reuse this text encoder as a fixed embedding extractor.
clip.eval()
for p, t in pairs:
    with torch.no_grad():
        emb = clip.text_enc(pad_token_lists([tok.encode(t)], pad_id=tok.pad).to(device))
    print(f"text: {t!r} -> embedding shape: {tuple(emb.shape)}")

### **3.3 Exercise: Build a Toy Music Generation System**
In this exercise, we will build a toy music generation system that uses a well-trained text encoder to extract text embeddings, and performs music generation using an LLM- and RVQ-based architecture.

In [ ]:
# ============================================================
# 3.3 Exercise: Conditional music generation using text embeddings
# ============================================================
# Reuse:
#   - RVQ codec from previous sections: `RVQ` (SimpleCodec)
#   - ARTransformer, NARTransformer, compute_losses
#
# We'll condition AR/NAR by *prefixing a learned projection* of the text embedding.
# This avoids introducing Transformer "memory" / cross-attention.

# Retrain RVQ on music data
latent_dim = 512
codebook_size = 100
num_quantizers = 4

lr = 1e-3
num_epochs = 200
intv  = 10

RVQ = SimpleCodec(latent_dim=latent_dim, codebook_size=codebook_size, num_quantizers=num_quantizers).cuda()
opt = torch.optim.Adam(RVQ.parameters(), lr=lr)

for i in range(num_epochs):
  audio_hat, vq_loss, _ = RVQ(audio)
  recon_loss = F.mse_loss(audio_hat, audio)
  loss = recon_loss + vq_loss

  opt.zero_grad()
  loss.backward()
  opt.step()

  if (i+1) % intv == 0:
    print(f"Epoch={i} recon_loss={recon_loss.item():.4f}  vq_loss={vq_loss.item():.4f}  total={loss.item():.4f}")

In [ ]:
# --- Sanity check: RVQ codec exists (trained earlier in the notebook) ---
assert "RVQ" in globals(), "Please run Section 2.* first to train RVQ (SimpleCodec), so `RVQ` exists."

RVQ = RVQ.to(device).eval()  # keep codec fixed for this exercise

# 1) Convert audio -> target codes using the codec (teacher targets)
with torch.no_grad():
    _, _, idx_list = RVQ(audio)              # idx_list: list length M, each (B,Tc)
    target_codes = torch.stack(idx_list, 1)  # (B,M,Tc)

B, M, Tc = target_codes.shape
print("target_codes:", target_codes.shape, "Tc frames per clip")

# 2) Build conditioning vector from trained CLIP text encoder
with torch.no_grad():
    text_emb = clip.text_enc(text_ids)  # (B, D_emb)
print("text_emb:", text_emb.shape)

# 3) Train AR+NAR models conditioned on the text embedding (prefix token)
code_vocab = 100  # codebook size
d_model = 256

ar = ARTransformer(vocab_size=code_vocab, d_model=d_model, max_len=10 + Tc).to(device)
nars = nn.ModuleList([NARTransformer(vocab_size=code_vocab, d_model=d_model, max_len=10 + Tc).to(device) for _ in range(M - 1)])

# Project text embedding -> model dimension (conditioning prefix)
cond_proj = nn.Linear(text_emb.size(-1), d_model).to(device)

params = list(ar.parameters()) + list(nars.parameters()) + list(cond_proj.parameters())
opt_gen = torch.optim.AdamW(params, lr=1e-3)

def ar_logits_with_prefix(ar_model, cond_vec, code_in):
    """AR forward: prefix 1 conditioning token, then code tokens. Return logits for code positions."""
    B, T = code_in.shape
    prefix = cond_vec.unsqueeze(1)  # (B,1,D)
    code_h = ar_model.tok(code_in)  # (B,T,D)
    x = torch.cat([prefix, code_h], dim=1)  # (B,1+T,D)
    x = x + ar_model.pe[None, :x.shape[1], :].to(code_in.device)
    S = 1 + T
    causal = torch.triu(torch.ones(S, S, device=code_in.device), diagonal=1).bool()
    h = ar_model.tr(x, mask=causal)
    return ar_model.head(h[:, 1:, :])  # (B,T,V)

def nar_logits_with_prefix(nar_model, cond_vec, coarse_codes):
    """NAR forward: prefix 1 conditioning token, then coarse codes as input. Parallel outputs."""
    B, T = coarse_codes.shape
    prefix = cond_vec.unsqueeze(1)                   # (B,1,D)
    code_h = nar_model.tok(coarse_codes)            # (B,T,D)
    x = torch.cat([prefix, code_h], dim=1)           # (B,1+T,D)
    x = x + nar_model.pe[None, :1+T, :].to(code_h.device)
    h = nar_model.tr(x)                              # no mask
    return nar_model.head(h[:, 1:, :])               # (B,T,V)

# --- training loop (toy) ---
ar.train(); nars.train(); cond_proj.train()
for step in range(400):
    cond = cond_proj(text_emb)  # (B,d_model)

    c1 = target_codes[:, 0, :]  # (B,Tc)
    bos = torch.ones(B, 1, dtype=torch.long, device=device) * ar.bos_id
    eos = torch.ones(B, 1, dtype=torch.long, device=device) * ar.eos_id
    c_in = torch.cat([bos, c1], dim=1)

    # AR loss
    logits1 = ar_logits_with_prefix(ar, cond, c_in)  # (B,Tc,V)
    loss_ar = F.cross_entropy(logits1.reshape(-1, logits1.size(-1)), torch.cat([c1, eos], dim=1).reshape(-1))

    # NAR residuals
    loss_nar = 0.0
    for m in range(1, M):
        logitsm = nar_logits_with_prefix(nars[m - 1], cond, c1)
        targetm = target_codes[:, m, :]
        loss_nar = loss_nar + F.cross_entropy(logitsm.reshape(-1, logitsm.size(-1)), targetm.reshape(-1))

    loss = loss_ar + loss_nar

    opt_gen.zero_grad()
    loss.backward()
    opt_gen.step()

    if (step + 1) % 100 == 0:
        print(f"step {step+1:03d} | loss={loss.item():.4f} | AR={loss_ar.item():.4f} | NAR={loss_nar.item():.4f}")



In [ ]:
# 4) Inference: generate codes given NEW text prompts, then decode to waveform
@torch.no_grad()
def generate_music(prompt: str, seconds=6.0):
    # 1) text -> embedding
    ids = pad_token_lists([tok.encode(prompt, max_len=96)], pad_id=tok.pad).to(device)
    emb = clip.text_enc(ids)                 # (1, D_emb)
    cond = cond_proj(emb)                    # (1, d_model)

    # 2) AR coarse
    T_out = int(SR * seconds)
    Tc_out = math.ceil(T_out / 256)  # because codec total downsampling is 256 (=4*8*8)
    # NOTE: if you change codec strides, update this factor.
    c = torch.zeros(1, 1, dtype=torch.long, device=device)  # BOS
    for _ in range(Tc_out - 1):
        logits = ar_logits_with_prefix(ar, cond, c)[:, -1, :]
        nxt = torch.argmax(logits, dim=-1, keepdim=True)
        c = torch.cat([c, nxt], dim=1)
    c1_gen = c  # (1,Tc_out)

    # 3) NAR residuals
    residuals = []
    for nar in nars:
        logits = nar_logits_with_prefix(nar, cond, c1_gen)
        residuals.append(torch.argmax(logits, dim=-1))
    codes = torch.stack([c1_gen] + residuals, dim=1)  # (1,M,Tc_out)

    # 4) decode codes -> waveform
    wav_hat = RVQ.decode_from_codes(codes)  # (1,1,~T)
    wav_hat = wav_hat[..., :T_out] if wav_hat.shape[-1] >= T_out else F.pad(wav_hat, (0, T_out - wav_hat.shape[-1]))
    return wav_hat.squeeze(0).squeeze(0).cpu().numpy()

# Try 3 prompts (can be unrelated to the tiny training set!)
test_prompts = [
    "classical orchestra",
    "upbeat guitar song",
    "soft piano melody",
]

for p in test_prompts:
    print("Prompt:", p)
    y = generate_music(p, seconds=6.0)
    print(y.shape)

### **3.4 Try A Commercial Example: Suno**

https://suno.com/create